In [290]:
import os
import csv
import pandas as pd
from metapub import PubMedFetcher

In [291]:
fetcher = PubMedFetcher()

In [393]:
df = pd.read_csv(os.path.join('..','resources','database','database.csv'))

In [394]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 749 entries, 0 to 748
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   title        749 non-null    object 
 1   authors_str  745 non-null    object 
 2   doi          743 non-null    object 
 3   abstract     657 non-null    object 
 4   pmid         749 non-null    int64  
 5   score        726 non-null    float64
 6   selected     726 non-null    object 
 7   translated   661 non-null    object 
 8   chosen       749 non-null    bool   
dtypes: bool(1), float64(1), int64(1), object(6)
memory usage: 47.7+ KB


In [378]:
df = df.drop_duplicates()

In [379]:
df = df.sort_values('score', ascending=False)
df = df.drop_duplicates(subset=['pmid'], keep='first')

In [380]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 749 entries, 209 to 1008
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   title        749 non-null    object 
 1   authors_str  745 non-null    object 
 2   doi          743 non-null    object 
 3   abstract     657 non-null    object 
 4   pmid         749 non-null    int64  
 5   score        726 non-null    float64
 6   selected     726 non-null    object 
 7   translated   661 non-null    object 
 8   chosen       749 non-null    bool   
dtypes: bool(1), float64(1), int64(1), object(6)
memory usage: 53.4+ KB


In [381]:
df.to_csv(os.path.join('..','resources','database','database.csv'), index=False)

In [382]:
df = pd.read_csv(os.path.join('..','resources','database','database.csv'))

In [376]:
with open ('chosen.txt', 'r') as file:
    chosen = file.readlines()

In [312]:
def search_pubmed(query):
    pmids = fetcher.pmids_for_query(f'({query}[Title])')
    print(f'({query}[Title])')
    if len(pmids) != 0:
        return pmids

In [299]:
chosen_clean = list()
for line in chosen:
    chosen_clean.append(line.strip())

In [300]:
chosen_clean = [line for line in chosen_clean if line != '']

In [301]:
chosen_pmids = list()
for title in chosen_clean:
    pmid = search_pubmed(title)
    if pmid != None:
        chosen_pmids.append(pmid[0])

(Actigraphy methodology in the Kids Mod PAH trial Physical activity as a functional endpoint in pediatric clinical trials[Title])
(Comparative adherence of macitentan versus ambrisentan and bosentan in Australian patients with pulmonary arterial hypertension  a retrospective real-[Title])
(Comparison Between REVEAL Lite 2 and COMPERA 2.0 for Risk Stratification in Pulmonary Arterial Hypertension[Title])
(Comparison of contemporary risk scores in all groups of pulmonary hypertension - a PVRI GoDeep meta-registry analysis[Title])
(Counterpoint Should the Use of Upfront Triple Combination Therapy Be Standard of Care in Pulmonary Arterial Hypertension No[Title])
(Evaluating the efficacy and safety of oral triple sequential combination therapy for treating patients with pulmonary arterial hypertension A multicenter retrospective study[Title])
(Evaluation of the European Society of Cardiology  ESC risk assessment score in incident systemic sclerosis-associated pulmonary arterial hypertension

In [383]:
chosen_int_pmids = set([int(pmid) for pmid in chosen_pmids])

In [384]:
df.loc[df['pmid'].isin(chosen_int_pmids), 'chosen'] = True
df.loc[~df['pmid'].isin(chosen_int_pmids), 'chosen'] = False

In [395]:
df[df['chosen']==True].info()

<class 'pandas.core.frame.DataFrame'>
Index: 41 entries, 2 to 748
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   title        41 non-null     object 
 1   authors_str  41 non-null     object 
 2   doi          40 non-null     object 
 3   abstract     36 non-null     object 
 4   pmid         41 non-null     int64  
 5   score        18 non-null     float64
 6   selected     18 non-null     object 
 7   translated   40 non-null     object 
 8   chosen       41 non-null     bool   
dtypes: bool(1), float64(1), int64(1), object(6)
memory usage: 2.9+ KB


In [386]:
chosen_in_df = list(df[df['chosen']==True]['pmid'].tolist())

In [387]:
chosen_to_add = [pmid for pmid in chosen_int_pmids if pmid not in chosen_in_df]
len(chosen_to_add)

0

In [388]:
list(df.columns)

['title',
 'authors_str',
 'doi',
 'abstract',
 'pmid',
 'score',
 'selected',
 'translated',
 'chosen']

In [389]:
def fetch_articles(pmids_list):
    art_list = list()
    for pmid in pmids_list:
        try:
            article = fetcher.article_by_pmid(pmid)
            art_list.append(article)
        except:
            print(f'Metapub exception: PMID {int(pmid)} not found')
    return art_list

In [390]:
def transform_article_list(art_list):
    transformed_art_list = list()
    for __art in art_list:
        __art_dic = dict()
        __art_dic['title'] = __art.title
        __art_dic['authors_str'] = __art.authors_str
        __art_dic['doi'] = __art.doi
        if __art.abstract:
            __art_dic['abstract'] = __art.abstract
        else:
            __art_dic['abstract'] = None
        __art_dic['pmid'] = __art.pmid
        __art_dic['score'] = None
        __art_dic['selected'] = None
        __art_dic['translated'] = False
        __art_dic['chosen'] = True
        transformed_art_list.append(__art_dic)
    return transformed_art_list

In [391]:
def save_to_database(art_list, columns):
    __csv_file = open(os.path.join('..','resources','database','database.csv'), 'a', newline='', encoding='utf-8')
    __writer = csv.DictWriter(__csv_file, list(columns))
    for __dic in art_list:
        __writer.writerow(__dic)
    __csv_file.close()

In [392]:
art_list = fetch_articles(chosen_to_add)
transformed_art_list = transform_article_list(art_list)
save_to_database(transformed_art_list, df.columns)